# UJIIndoorLoc trên Google Colab

Notebook này tải mã từ một nhánh GitHub, tự liệt kê model và cách xử lý RSS, chạy một hoặc nhiều model rồi hiển thị bảng và biểu đồ ngay trong Colab. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy BSRBF-KAN. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation. Ma trận nhầm lẫn chỉ áp dụng cho dự đoán tầng của KNN/WKNN; tọa độ dùng sai số mét và CDF.


In [ ]:
import io
import zipfile
import urllib.error
import urllib.request
from pathlib import Path
import subprocess
import shutil
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "main"  # đổi thành feature/data-robustness hoặc feature/kan-mixer để thử PR
root = Path("/content/repo_source")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if token:
    url = f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}"
    try:
        request = urllib.request.Request(
            url, headers={"Authorization": f"Bearer {str(token).strip()}",
                          "Accept": "application/vnd.github+json"})
        with urllib.request.urlopen(request, timeout=120) as response:
            repo_zip = response.read()
    except urllib.error.HTTPError as error:
        print(f"GitHub trả HTTP {error.code}. Kiểm tra token có quyền Contents: Read,"
              " đã chọn đúng repo, bật quyền truy cập Secret cho notebook và BRANCH tồn tại.")
        repo_zip = None
    except urllib.error.URLError:
        print("Không kết nối được GitHub API. Có thể dùng ZIP của repo để tiếp tục.")
        repo_zip = None
else:
    repo_zip = None
token = None
if repo_zip is None:
    print("Upload ZIP của nhánh GitHub bạn muốn chạy (Code → Download ZIP).")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP của repo")
    repo_zip = next(iter(uploaded.values()))

if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(repo_zip)) as z:
    z.extractall(root)
project_files = list(root.rglob("pyproject.toml"))
if len(project_files) != 1:
    raise RuntimeError(f"Cần đúng một pyproject.toml; tìm thấy {len(project_files)}")
project = project_files[0].parent
print("Project:", project)
subprocess.run(["python", "-m", "pip", "install", "-e", str(project)], check=True)


In [ ]:
import shutil
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name}")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


## Chọn model và tiền xử lý

Danh sách dưới đây đọc trực tiếp từ các file plugin trong nhánh đã tải, không cần sửa notebook. Nếu nhánh có họ KNN/WKNN, mục **Họ model** sẽ xuất hiện. Sau khi thành viên push file mới, đổi `BRANCH` ở ô đầu rồi chạy lại ô tải repo và ô chọn này. Chọn preset smoke để thử luồng trước. Preset Mixer chỉ chạy với `bsrbf_kan` và `rss_standard` (520 AP). Một cấu hình đầy đủ có thể mất nhiều giờ.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import json

options = json.loads(subprocess.check_output(
    ["python", "-m", "ips.experiments.cli", "--list-options"], text=True))
presets = sorted(str(p.relative_to(project)) for p in (project / "configs").glob("*.json"))
families = {"Neural": options["models"]}
if (project / "src/ips/neighbors").is_dir():
    neighbor_probe = subprocess.run(
        ["python", "-c", "import json; from ips.neighbors import available_models; print(json.dumps(sorted(available_models())))"],
        capture_output=True, text=True)
    if neighbor_probe.returncode:
        raise RuntimeError(f"Không tải được plugin KNN: {neighbor_probe.stderr}")
    families["KNN/WKNN"] = json.loads(neighbor_probe.stdout)
family_choice = widgets.Dropdown(options=list(families), description="Họ model:")
model_choice = widgets.SelectMultiple(options=families["Neural"], description="Models:", value=("mlp",) if "mlp" in options["models"] else ())
preprocess_choice = widgets.Dropdown(options=options["preprocessors"], description="RSS:", value="rss_standard" if "rss_standard" in options["preprocessors"] else None)
neural_presets = [p for p in presets if not Path(p).name.startswith("knn_")]
knn_presets = [p for p in presets if Path(p).name.startswith("knn_")]
preset_choice = widgets.Dropdown(options=neural_presets, description="Preset:", value="configs/mlp_smoke.json" if "configs/mlp_smoke.json" in neural_presets else None)
preprocess_params_choice = widgets.Text(value="{}", description="RSS params:", placeholder='{"min_observations": 20}')

def change_family(change):
    is_knn = change["new"] == "KNN/WKNN"
    model_choice.options = families[change["new"]]
    model_choice.value = (families[change["new"]][0],) if families[change["new"]] else ()
    preset_choice.options = knn_presets if is_knn else neural_presets
    if is_knn and "rss_powed" in options["preprocessors"]:
        preprocess_choice.value = "rss_powed"
    elif not is_knn and "rss_standard" in options["preprocessors"]:
        preprocess_choice.value = "rss_standard"

family_choice.observe(change_family, names="value")
display(family_choice, model_choice, preprocess_choice, preprocess_params_choice, preset_choice)
print("Models:", families, "| Preprocessors:", options["preprocessors"])


In [ ]:
import torch
from datetime import datetime, timezone

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
selected_models = list(model_choice.value)
selected_preprocess = preprocess_choice.value
selected_preset = preset_choice.value
preprocess_params = json.loads(preprocess_params_choice.value)
if not isinstance(preprocess_params, dict):
    raise ValueError("RSS params phải là JSON object, ví dụ {}")
if not selected_models or not selected_preprocess or not selected_preset:
    raise ValueError("Hãy chọn model, RSS và preset")
settings = json.loads((project / selected_preset).read_text())
print('Chạy:', selected_models, '| Tiền xử lý:', selected_preprocess, '| Preset:', selected_preset)
print('Cấu hình preset:', json.dumps(settings, ensure_ascii=False, indent=2))
print('Số dòng dữ liệu:', [(p.name, sum(1 for _ in p.open()) - 1) for p in data_dir.glob('*.csv')])
from ips.experiments.visualize import render_run, compare_runs
run_dirs = []
if family_choice.value == "KNN/WKNN":
    settings.update(models=selected_models, preprocess=selected_preprocess,
                    preprocess_params=preprocess_params)
    selected_config = Path("/content") / f"selected_{RUN_NAME}.json"
    selected_config.write_text(json.dumps(settings), encoding="utf-8")
    subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                    "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
    render_run(run_dir)
    run_dirs.append(run_dir)
else:
    if settings.get("with_mixer") and (selected_models != ["bsrbf_kan"] or selected_preprocess != "rss_standard"):
        raise ValueError("Preset Mixer cần model bsrbf_kan và RSS rss_standard")
    print("GPU available:", torch.cuda.is_available())
    if "bsrbf_kan" in selected_models and not torch.cuda.is_available():
        print("Cảnh báo: BSRBF-KAN trên CPU có thể rất chậm.")
    for selected_model in selected_models:
        model_dir = run_dir / selected_model
        print('Đang chạy:', selected_model)
        subprocess.run(["python", "-m", "ips.experiments.cli", "--config", str(project / selected_preset),
                        "--model", selected_model, "--preprocess", selected_preprocess,
                        "--preprocess-params", json.dumps(preprocess_params),
                        "--data-dir", str(data_dir), "--output-dir", str(model_dir)], check=True)
        render_run(model_dir)
        run_dirs.append(model_dir)
    if len(run_dirs) > 1:
        compare_runs(run_dirs, labels=selected_models)
print('Kết quả:', run_dir)


In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
